# Trovare i driver della domanda con PROC GLMSELECT: selezione stepwise, LASSO e selezione forward validata

## Sintesi esecutiva

Un team di analytics retail usa **PROC GLMSELECT** per scoprire quali leve di marketing e prezzo muovono davvero le vendite settimanali in unità, separando i veri driver della domanda dal rumore. La selezione stepwise valutata con SBC, il LASSO con validazione incrociata a 5 fold e una ricerca forward validata su un holdout recuperano tutte lo **stesso insieme di veri driver** — prezzo proprio, prezzo del concorrente, spesa pubblicitaria, volume email, promozioni, festivi, la regione Northeast e il canale Online — e ognuna delle quattro esche piantate (`temp_f`, `noise1`, `noise2`, `noise3`) viene scartata automaticamente.

I metodi concordano anche da vicino sulle magnitudini: ciascuno stima un effetto del prezzo proprio vicino a **-28 unità per dollaro** e un effetto del prezzo del concorrente vicino a **+14**, i segni di sostituzione incorporati nell'equazione generatrice dei dati. L'unico disaccordo è al margine — il LASSO validato con cross-validation mantiene inoltre un piccolo contrasto `Region=Midwest` statisticamente trascurabile (stima -8.3 con un errore standard di 8.3) che sia la selezione stepwise sia quella forward scartano. Un elenco di driver che sopravvive a tre filosofie di selezione diverse è molto più affidabile di uno calibrato su una singola regola.

## Fonti dei dati

Tutti i dati in questo notebook sono **sintetici** e generati inline (nessun file esterno, seed `20260531`). Riproducono una stagione di pannelli di domanda negozio-settimana per un rivenditore di beni di consumo.

| Dataset | Righe | Grana | Variabili chiave |
|---------|------|-------|---------------|
| `demand` | 100 | Negozio-settimana | `units` (risposta: unità vendute settimanalmente); `price`, `competitor` (prezzo proprio e del rivale a scaffale); `adspend`, `email` (pressione media); `promo`, `holiday` (flag di evento); `region`, `channel` (effetti CLASS); `temp_f`, `noise1`-`noise3` (predittori esca/irrilevanti) |

`units` è costruita a partire da un'equazione lineare nota, così che l'insieme "corretto" di driver sia verificabile; `temp_f` e le tre colonne `noise` non portano alcun segnale reale ed esistono per verificare se ogni metodo di selezione le scarta.

# Trovare i driver della domanda con PROC GLMSELECT

Un category manager ha decine di variabili esplicative candidate per le vendite settimanali: prezzo proprio, prezzo del concorrente, spesa pubblicitaria, volume email, promozioni, festivi, regione del negozio, canale di vendita, persino il meteo. Inserirle tutte in un'unica regressione favorisce l'overfitting e coefficienti instabili. **PROC GLMSELECT** automatizza la ricerca di un modello parsimonioso, supportando la selezione stepwise, LASSO, elastic net e least-angle con validazione incrociata e partizionamento holdout integrati.

In questo notebook:

1. Generiamo un pannello di domanda negozio-settimana sintetico e realistico con un insieme *noto* di veri driver (più variabili esca deliberate).
2. Eseguiamo la **selezione stepwise** valutata con SBC.
3. Eseguiamo il **LASSO** con validazione incrociata a 5 fold.
4. Eseguiamo la **selezione forward validata su un holdout del 30%**.

Un buon metodo di selezione dovrebbe recuperare i driver reali e scartare il rumore — vediamo.

## 1. Generare un pannello di domanda sintetico

La risposta `units` è costruita a partire da un'equazione lineare esplicita, quindi conosciamo la verità di riferimento: prezzo e prezzo del concorrente, spesa pubblicitaria, volume email, i flag promo e festivo, oltre agli effetti di regione e canale, contano tutti. Le variabili `temp_f`, `noise1`, `noise2` e `noise3` sono pure esche senza alcuna relazione reale con le vendite. Un seed di `call streaminit` rende i dati riproducibili.

In [1]:
/* ---------------------------------------------------------------
   Pannello sintetico di domanda negozio-settimana per un rivenditore di beni di consumo.
   'units' segue un'equazione NOTA; temp_f e noise1-3 sono esche.
   --------------------------------------------------------------- */
DATI demand;
    CHIAMARE streaminit(20260531);
    LUNGHEZZA region $9 channel $8 promo $3;
    FARE store_week = 1 FINO_A 100;
        /* Mix di regioni */
        u1 = rand('uniform');
        SE_COND u1 < 0.34 ALLORA region = 'Northeast';
        ALTRIMENTI SE_COND u1 < 0.67 ALLORA region = 'Midwest';
        ALTRIMENTI region = 'South';

        /* Canale di vendita */
        SE_COND rand('uniform') < 0.45 ALLORA channel = 'Store';
        ALTRIMENTI channel = 'Online';

        /* Driver di prezzo e media */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Flag di evento e un'esca meteo irrilevante */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        SE_COND rand('uniform') < 0.40 ALLORA promo = 'Yes';
        ALTRIMENTI promo = 'No';

        /* Predittori di puro rumore (nessun effetto reale) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Unità vendute settimanalmente da un'equazione strutturale nota */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        SE_COND units < 0 ALLORA units = 0;
        USCITA;
    FINE;
    MANTENERE region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
ESEGUIRE;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Profilare i dati

Prima della modellazione, verifichiamo che la risposta e i principali candidati continui siano su scale sensate.

In [2]:
PROCEDURA MEDIE DATI=demand n mean std min max maxdec=1;
    VARIABILE units price competitor adspend email;
    ETICHETTA units="Unità vendute (settimanali)" price="Prezzo proprio"
          competitor="Prezzo del concorrente" adspend="Spesa pubblicitaria"
          email="Volume email";
    TITOLO "Domanda settimanale e driver candidati";
ESEGUIRE;

                                         Domanda settimanale e driver candidati                                         

                                                  The MEANS Procedure

 Variable    Label                                N        Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------------------------------------
 units       Unità vendute (settimanali)        100       874.8       136.3       508.6      1162.3
 price       Prezzo proprio                     100        14.0         3.4         8.0        19.9
 competitor  Prezzo del concorrente             100        13.8         3.4         8.1        19.9
 adspend     Spesa pubblicitaria                100       990.6       726.9        50.0      3358.0
 email       Volume email                       100        45.5        26.4         0.0        99.0
 --------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Selezione stepwise valutata con SBC

La selezione stepwise aggiunge e rimuove effetti alternativamente, qui giudicata dal **Criterio Bayesiano di Schwarz (SBC)** sia per il test di ingresso (`select=sbc`) sia per la scelta finale del modello (`choose=sbc`). SBC penalizza la complessità più pesantemente dell'AIC, favorendo modelli più snelli.

Istruzioni e opzioni chiave:

- `CLASS region channel promo / param=reference` dichiara i predittori categoriali con codifica a cella di riferimento.
- `selection=stepwise(select=sbc choose=sbc)` guida la ricerca.
- `details=summary` stampa il riepilogo passo-passo della selezione; `stb` aggiunge i coefficienti standardizzati così che gli effetti su scale diverse siano confrontabili.
- `output out=demand_scored p=predicted r=residual` salva i valori stimati e i residui per il punteggio a valle.

Leggi il **riepilogo della selezione stepwise** come la traccia della ricerca: parte dal modello completo a 12 effetti e *rimuove* gli effetti uno alla volta, scartando in successione `noise1`, `noise2`, `temp_f`, il contrasto `Region=Midwest` e `noise3`, perché ogni rimozione abbassa l'SBC. Ciò che sopravvive nella tabella delle **stime dei parametri** è il modello scelto.

In [3]:
PROCEDURA glmselect DATI=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODELLO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    ETICHETTA units="Unità vendute (settimanali)" region="Regione" channel="Canale"
          promo="Promozione" price="Prezzo proprio" competitor="Prezzo del concorrente"
          adspend="Spesa pubblicitaria" email="Volume email"
          temp_f="Temperatura (°F)" holiday="Festivo"
          noise1="Rumore 1" noise2="Rumore 2" noise3="Rumore 3";
    USCITA out=demand_scored p=predicted r=residual;
    TITOLO "Selezione stepwise dei driver della domanda (SBC)";
ESEGUIRE;

                                         Domanda settimanale e driver candidati                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Unità vendute (settimanali)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                  Stepwise Selection Summary                                                  

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  -----------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed           Rumore 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.87


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO con validazione incrociata

Il LASSO restringe i coefficienti verso zero, effettuando selezione e regolarizzazione simultaneamente. Invece di fermarci a un criterio fisso, lasciamo che la **validazione incrociata a 5 fold** scelga il punto sul percorso LASSO con il miglior errore fuori campione.

- `selection=lasso(choose=cv stop=none)` traccia l'intero percorso LASSO e seleziona il passo ottimale in CV.
- `cvmethod=random(5)` assegna le osservazioni a 5 fold casuali.

Il **riepilogo della selezione LASSO** mostra l'ordine in cui gli effetti entrano man mano che la penalità si allenta: prima `price`, poi `adspend`, `competitor`, la regione Northeast, `email`, `promo` e `holiday` — tutti e sette i segnali veri entrano prima di qualsiasi esca. La validazione incrociata sceglie quindi il passo in cui il PRESS fuori campione è più basso, e la tabella delle **stime dei parametri** risultante mantiene esattamente i driver genuini (più un piccolo termine `Region=Midwest`) escludendo `temp_f`, `noise1`, `noise2` e `noise3`, che entrano solo alla fine del percorso.

In [4]:
PROCEDURA glmselect DATI=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODELLO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=random(5);
    ETICHETTA units="Unità vendute (settimanali)" region="Regione" channel="Canale"
          promo="Promozione" price="Prezzo proprio" competitor="Prezzo del concorrente"
          adspend="Spesa pubblicitaria" email="Volume email"
          temp_f="Temperatura (°F)" holiday="Festivo"
          noise1="Rumore 1" noise2="Rumore 2" noise3="Rumore 3";
    TITOLO "LASSO con validazione incrociata a 5 fold";
ESEGUIRE;

                                         Domanda settimanale e driver candidati                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Unità vendute (settimanali)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                            LASSO Selection Summary                                                             

    Step    Action                  Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  -------------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Selezione forward validata su un holdout

Una terza strategia complementare: costruire il modello per **selezione forward** (gli effetti entrano soltanto, non escono mai), ma fermarsi dove la performance su un campione holdout indipendente è migliore — proteggendo direttamente dall'overfitting.

- `partition fraction(validate=0.30)` riserva casualmente il 30% delle righe per la validazione, lasciando 70 osservazioni di training e 30 di validazione.
- `selection=forward(select=aic choose=validate)` aggiunge effetti per AIC ma sceglie il modello finale in base all'errore sul campione di validazione.

La tabella delle **frazioni di partizione** conferma la suddivisione 70/30. La selezione forward inserisce quindi effetti finché l'errore di validazione smette di migliorare; gli otto effetti nella tabella finale delle **stime dei parametri** sono esattamente i veri driver, mentre le quattro esche non vengono mai ammesse. Quando tre metodi costruiti su principi diversi arrivano agli stessi driver, il modello è molto più probabilmente reale che un artefatto di una singola regola di selezione.

In [5]:
PROCEDURA glmselect DATI=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODELLO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition fraction(validate=0.30);
    ETICHETTA units="Unità vendute (settimanali)" region="Regione" channel="Canale"
          promo="Promozione" price="Prezzo proprio" competitor="Prezzo del concorrente"
          adspend="Spesa pubblicitaria" email="Volume email"
          temp_f="Temperatura (°F)" holiday="Festivo"
          noise1="Rumore 1" noise2="Rumore 2" noise3="Rumore 3";
    TITOLO "Selezione forward validata su un holdout del 30%";
ESEGUIRE;

                                         Domanda settimanale e driver candidati                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Unità vendute (settimanali)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                           Forward Selection Summary                                                            

    Step    Action                  Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  ------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interpretare i risultati

Tutte e tre le strategie di selezione recuperano lo **stesso insieme di veri driver della domanda** e scartano ogni esca. Leggendo direttamente dalle tabelle delle **stime dei parametri**:

- **Il prezzo** è il driver dominante. Il suo coefficiente standardizzato nel modello stepwise è **-0.70**, di gran lunga il più grande in magnitudine, e la pendenza grezza si colloca tra **-27.8** (stepwise e LASSO) e **-28.6** (forward) unità per dollaro — quasi esattamente il -28 con cui i dati sono stati generati. **Il prezzo del concorrente** muove la domanda nella direzione opposta di circa **+14.4** in tutti e tre i fit, il comportamento di sostituzione che un category manager si aspetta.
- **La spesa pubblicitaria** (circa +0.062 unità per dollaro) e il **volume email** (circa +1.2 unità per invio) aumentano entrambe le vendite, quantificando la risposta ai media.
- **Le promozioni** e i **festivi** portano i maggiori effetti di evento: il contrasto `Promo=No` è di circa **-51 a -57** rispetto a una settimana in promozione, e l'aumento nei festivi è di circa **+66 a +76** unità. La **regione Northeast** (circa +49 a +55) e il **canale Online** (circa +24 a +32) portano effetti di baseline positivi.
- In modo cruciale, ogni esca — `temp_f`, `noise1`, `noise2`, `noise3` — viene **scartata** dalla selezione stepwise e forward ed è esclusa dal modello LASSO scelto tramite CV. Ogni metodo ha recuperato il segnale strutturale e ignorato il rumore.

I tre metodi non sono identici byte per byte: la selezione stepwise (SBC) e la ricerca forward validata su holdout si assestano sugli stessi otto effetti, mentre il LASSO validato con cross-validation mantiene inoltre un piccolo contrasto `Region=Midwest` (-8.3, errore standard 8.3) — una differenza al livello del rumore più che un disaccordo sostanziale. Che un elenco di driver sopravviva alla selezione stepwise SBC, al LASSO validato con cross-validation e alla selezione forward validata su holdout è il vero punto: un risultato robusto a tre filosofie di selezione diverse è molto più credibile di uno calibrato su un singolo criterio. Con `OUTPUT OUT=demand_scored` che cattura i valori stimati e i residui, lo stesso flusso di lavoro si estende naturalmente al punteggio del calendario di prezzi e promozioni pianificato per il prossimo trimestre.